# 🚀 PBL6 TVPR - SpaceTime-DAMR Pipeline Notebook (Google Colab)
Thư mục mã nguồn: `srccv` | Divided Space-Time Vision Transformer + BERT for Text-to-Video Person Retrieval  
Kiến trúc thuần ViT + BERT (Không dùng pretrain CLIP, so sánh công bằng chuẩn paper ACM MM '24) | TVPReid Benchmark

In [ ]:
# ==============================================================================
# CELL 1: MOUNT GOOGLE DRIVE & MULTI-DATASET SETUP (srccv - SpaceTime-DAMR)
# ==============================================================================
!pip install -q transformers pyyaml

import os
import sys
import shutil
import zipfile

# 1. Mount Google Drive
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/pbl6remake"
else:
    DRIVE_PROJECT_ROOT = os.path.abspath(".")

# OPTION FLAG: Copy/Extract datasets to Colab local SSD (/content/dataset) for max speed
COPY_TO_LOCAL_SSD = True
LOCAL_DATASET_DIR = "/content/dataset"
os.makedirs(LOCAL_DATASET_DIR, exist_ok=True)

# 2. Automatically Extract all Zip archives & Copy dataset folders to Colab SSD
if COPY_TO_LOCAL_SSD and 'google.colab' in sys.modules:
    print("[*] Preparing datasets on Colab SSD (/content/dataset)...")

    # Danh sách các file zip cần tự động giải nén trên Drive
    zip_candidates = [
        "CUHK-PEDES.zip",
        "TVPReid_dataset.zip",
        "dataset.zip"
    ]

    for zname in zip_candidates:
        zpath = os.path.join(DRIVE_PROJECT_ROOT, zname)
        if os.path.exists(zpath):
            print(f"[*] Found zip archive: {zname}. Extracting to /content/dataset...")
            with zipfile.ZipFile(zpath, 'r') as zip_ref:
                zip_ref.extractall(LOCAL_DATASET_DIR)
            print(f"[+] Successfully extracted {zname} to Colab SSD!")

    # Danh sách các thư mục chưa nén trên Drive (fallback)
    dir_candidates = ["CUHK-PEDES", "TVPReid_dataset"]
    for dname in dir_candidates:
        dpath = os.path.join(DRIVE_PROJECT_ROOT, dname)
        target_path = os.path.join(LOCAL_DATASET_DIR, dname)
        if os.path.exists(dpath) and not os.path.exists(target_path):
            print(f"[*] Copying folder {dname} from Drive to Colab SSD...")
            shutil.copytree(dpath, target_path, dirs_exist_ok=True)
            print(f"[+] Successfully copied {dname} to Colab SSD!")

    DATA_ROOT = LOCAL_DATASET_DIR
else:
    print("[*] Mode: Reading dataset directly from project directory.")
    DATA_ROOT = DRIVE_PROJECT_ROOT

# 3. Add srccv directory to sys.path for direct module imports
SRC_DIR = os.path.join(DRIVE_PROJECT_ROOT, "srccv")
if not os.path.exists(SRC_DIR):
    SRC_DIR = os.path.abspath("srccv") if os.path.exists("srccv") else DRIVE_PROJECT_ROOT
if os.path.exists(SRC_DIR) and SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f"\n[+] Current DATA_ROOT: {DATA_ROOT}")
print(f"[+] Current SRC_DIR (srccv): {SRC_DIR}")

In [ ]:
# ==============================================================================
# CELL 2: EXECUTE SPACETIME-DAMR TRAINING PIPELINE (Pure ViT + BERT)
# ==============================================================================
import os
import sys
import shutil
import warnings
import logging
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
# ==============================================================================
import yaml
from pathlib import Path

# 1. Đường dẫn thư mục mã nguồn srccv chuẩn xác (tránh duplicate srccv/srccv)
if 'google.colab' in sys.modules:
    SRC_DIR = '/content/drive/MyDrive/pbl6remake/srccv'
else:
    cur = os.path.abspath('.')
    SRC_DIR = cur if os.path.basename(cur) == 'srccv' else os.path.join(cur, 'srccv')

# Tự động dọn dẹp thư mục srccv/srccv bị duplicate trước đây trên Google Drive nếu có
nested_srccv = os.path.join(SRC_DIR, 'srccv')
if os.path.exists(nested_srccv) and os.path.isdir(nested_srccv):
    print('[*] Phát hiện thư mục lồng duplicate srccv/srccv. Đang tự động dọn dẹp và di chuyển dữ liệu...')
    for folder_name in ['checkpoints', 'reports', 'outputs']:
        sub_nested = os.path.join(nested_srccv, folder_name)
        sub_target = os.path.join(SRC_DIR, folder_name)
        if os.path.exists(sub_nested):
            os.makedirs(sub_target, exist_ok=True)
            for item in os.listdir(sub_nested):
                s_path = os.path.join(sub_nested, item)
                d_path = os.path.join(sub_target, item)
                if not os.path.exists(d_path):
                    shutil.move(s_path, d_path)
    shutil.rmtree(nested_srccv, ignore_errors=True)
    print('[+] Đã dọn dẹp sạch sẽ thư mục duplicate srccv/srccv!')

config_path = os.path.join(SRC_DIR, 'configs', 'default.yaml')
LOCAL_DATASET = '/content/dataset'

# 2. CẤU HÌNH HUẤN LUYỆN & SUB-DATASET:
# ------------------------------------------------------------------------------
# Sub-dataset: 'TVPReid-Duke' (7,840 mẫu) | 'TVPReid-iLIDs' (976 mẫu) | 'TVPReid-PRID' (1,842 mẫu) | 'all'
SUB_DATASET = 'TVPReid-iLIDs'

# CHẾ ĐỘ HUẤN LUYỆN (TRAIN_MODE):
#   - 'new'    : Huấn luyện mới hoàn toàn từ đầu (Epoch 1, reset weights).
#   - 'latest' : Tự động nạp checkpoint gần nhất (latest.pth) của sub-dataset để tiếp tục train.
#   - 'best'   : Tự động nạp checkpoint tốt nhất (best.pth) của sub-dataset để tối ưu tiếp/fine-tune.
TRAIN_MODE = 'new'  # 'latest' để tiếp tục train từ epoch gần nhất, hoặc 'new' để train lại từ đầu

BATCH_SIZE = 16  # Mini-batch 8 + Accum 2 = Effective Batch Size 16 (Tiết kiệm VRAM cho 16 frames)
GRAD_ACCUM_STEPS = 1
LR = 0.0001
EPOCHS = 50
# ------------------------------------------------------------------------------

# 3. TỰ ĐỘNG ÁNH XẠ ĐƯỜNG DẪN CHECKPOINT THEO SUB_DATASET & TRAIN_MODE
def resolve_checkpoint(src_dir: str, sub_dataset: str, mode: str):
    """Tự động chuẩn hóa sub-dataset và tìm file checkpoint tương ứng."""
    mode_str = str(mode).strip().lower()
    if mode_str == 'new':
        return None, 'Bắt đầu huấn luyện từ đầu (Epoch 1, không nạp checkpoint).'

    sub_clean = str(sub_dataset).strip()
    low = sub_clean.lower()
    if low in ['all', 'combined', '*', 'both', 'none', '', 'tvpreid', 'tvpreid_dataset', 'tvp-reid', 'tvpreid-combined']:
        sub_folder = 'all'
    elif 'duke' in low:
        sub_folder = 'TVPReid-Duke'
    elif 'ilids' in low or 'i-lids' in low:
        sub_folder = 'TVPReid-iLIDs'
    elif 'prid' in low:
        sub_folder = 'TVPReid-PRID'
    elif 'cuhk' in low or 'pedes' in low:
        sub_folder = 'CUHK-PEDES'
    else:
        sub_folder = sub_clean

    ckpt_base = os.path.join(src_dir, 'checkpoints')
    sub_ckpt_dir = os.path.join(ckpt_base, sub_folder)

    candidates = []
    if mode_str == 'latest':
        candidates = [
            os.path.join(sub_ckpt_dir, 'latest.pth'),
            os.path.join(ckpt_base, 'latest.pth'),
        ]
    elif mode_str == 'best':
        candidates = [
            os.path.join(sub_ckpt_dir, 'best.pth'),
            os.path.join(sub_ckpt_dir, 'best_damr.pth'),
            os.path.join(ckpt_base, 'best.pth'),
            os.path.join(ckpt_base, 'best_damr.pth'),
        ]
    else:
        return None, f"Chế độ TRAIN_MODE '{mode}' không hợp lệ. Mặc định chạy 'new'."

    for cand in candidates:
        if os.path.exists(cand) and os.path.getsize(cand) > 0:
            return cand, f'Đã tìm thấy checkpoint [{mode}]: {cand}'

    expected_path = os.path.join(sub_ckpt_dir, 'latest.pth' if mode_str == 'latest' else 'best.pth')
    return None, f"Chưa có checkpoint '{mode}' tại ({expected_path}). Tự động chuyển sang chế độ 'new' (Epoch 1)."

target_checkpoint, ckpt_msg = resolve_checkpoint(SRC_DIR, SUB_DATASET, TRAIN_MODE)
resume_flag = f'--resume "{target_checkpoint}"' if target_checkpoint else ''

# 4. Tự động đồng bộ cấu hình vào default.yaml trước khi train
if os.path.exists(config_path):
    with open(config_path, 'r', encoding='utf-8') as f:
        cfg = yaml.safe_load(f) or {}
    if 'data' not in cfg: cfg['data'] = {}
    cfg['data']['sub_dataset'] = SUB_DATASET
    cfg['data']['batch_size'] = BATCH_SIZE
    if 'train' not in cfg: cfg['train'] = {}
    cfg['train']['grad_accum_steps'] = GRAD_ACCUM_STEPS
    if os.path.exists(LOCAL_DATASET):
        cfg['data']['data_root'] = LOCAL_DATASET
    with open(config_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, default_flow_style=False)

# Purge cached module để nạp cập nhật mới nhất
for mod in list(sys.modules.keys()):
    if mod.startswith('data.') or mod == 'data' or mod.startswith('utils.') or mod == 'utils':
        del sys.modules[mod]

%cd {SRC_DIR}
!find . -type d -name "__pycache__" -exec rm -rf {} + 2>/dev/null || true

data_root_arg = LOCAL_DATASET if os.path.exists(LOCAL_DATASET) else 'auto'

print('\n' + '=' * 70)
print('🚀 SPACETIME-DAMR TRAINING LAUNCHER')
print(f'  • Sub-dataset   : {SUB_DATASET}')
print(f'  • Train Mode    : {TRAIN_MODE.upper()}')
print(f'  • Checkpoint    : {target_checkpoint if target_checkpoint else "None (Start fresh)"}')
print(f'  • Status/Action : {ckpt_msg}')
print(f'  • Data root     : {data_root_arg}')
print(f'  • Batch Size    : {BATCH_SIZE} (Accum: {GRAD_ACCUM_STEPS} -> Effective: {BATCH_SIZE * GRAD_ACCUM_STEPS}) | LR: {LR} | Epochs: {EPOCHS}')
print('=' * 70 + '\n')

# 5. Chạy huấn luyện
!python train.py --config {config_path} --sub_dataset {SUB_DATASET} --data_root {data_root_arg} --batch_size {BATCH_SIZE} --grad_accum_steps {GRAD_ACCUM_STEPS} --lr {LR} --epochs {EPOCHS} --num_workers 2 {resume_flag}


In [ ]:
# ==============================================================================
# ⚡ CELL 3: 1-CELL ALL-IN-ONE DEMO & INFERENCE (SpaceTime-DAMR Standalone)
# ==============================================================================
import os
import sys
import warnings
import logging
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
# ==============================================================================
import torch
import numpy as np

print("[1/4] 📦 Checking & installing dependencies...")
!pip install -q transformers pyyaml

# 1. Mount Google Drive & Setup Path for srccv
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    PROJECT_ROOT = "/content/drive/MyDrive/pbl6remake"
    SRC_DIR = os.path.join(PROJECT_ROOT, "srccv")
    if os.path.exists(SRC_DIR) and SRC_DIR not in sys.path:
        sys.path.insert(0, SRC_DIR)
    os.chdir(SRC_DIR)
    print(f"[+] Working directory set to (srccv): {SRC_DIR}")
else:
    PROJECT_ROOT = os.path.abspath(".")
    SRC_DIR = os.path.join(PROJECT_ROOT, "srccv") if os.path.exists(os.path.join(PROJECT_ROOT, "srccv")) else PROJECT_ROOT
    if SRC_DIR not in sys.path:
        sys.path.insert(0, SRC_DIR)
    os.chdir(SRC_DIR)

# 2. Load SpaceTime-DAMR Model & Components (Pure ViT + BERT, 768-D)
print("\n[2/4] 🧠 Loading SpaceTime-DAMR Model (SpaceTimeViT + BERT)...\n")
from models.damr_main import DAMRModel
from utils.paths import paths
from data.dataset import create_dataloaders
from utils.metrics import compute_similarity_matrix, evaluate_rank_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Computing device: {device}")

model = DAMRModel(
    embed_dim=768,
    text_model_name="bert-base-uncased",
    motion_frame_num=16,
    vit_depth=6,
    vit_heads=8
).to(device)

checkpoint_path = paths.get_checkpoint_path("best.pth")

if os.path.exists(checkpoint_path):
    print(f"[+] Loaded checkpoint from {checkpoint_path}")
    try:
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    except TypeError:
        ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
else:
    print("⚠️ No checkpoint found. Using initialized SpaceTime-DAMR model weights.")

# 3. Test Full Natural Language Query Encoding & Single Query Inference
QUERY_TEXT = "A woman in a red jacket walking towards the gate"
print(f"\n[3/4] 🎯 Encoding Full Query via BERT (768-D): '{QUERY_TEXT}'...")

model.eval()
with torch.no_grad():
    T = model.encode_text([QUERY_TEXT], device=device)

print("\n[4/4] 📊 Evaluating retrieval on Validation Set...")
active_sub = getattr(paths, "sub_dataset", "TVPReid-Duke")
_, val_loader = create_dataloaders(
    data_root=str(paths.data_root),
    sub_dataset=active_sub,
    batch_size=16,
    num_frames=16
)

all_v = []
all_v_app = []
all_pids = []
video_ids = []

with torch.no_grad():
    for batch in val_loader:
        video_frames = batch.get("video_frames", batch.get("motion_frames", batch.get("keyframes"))).to(device, non_blocking=True)
        V_app, V_mot, V_video = model.encode_video(video_frames)
        all_v.append(V_video.cpu())
        all_v_app.append(V_app.cpu())
        all_pids.extend(batch["person_id"].numpy())
        video_ids.extend(batch["video_id"])

V = torch.cat(all_v, dim=0)
V_app = torch.cat(all_v_app, dim=0)

sim_matrix = compute_similarity_matrix(T.cpu(), V, V_app, beta=0.85)
scores = sim_matrix[0]
top_indices = np.argsort(-scores)[:5]

print("\n🏆 TOP-5 RETRIEVED PERSON VIDEOS FOR QUERY:")
for rank, idx in enumerate(top_indices, 1):
    print(f"  Rank #{rank}: Video ID = {video_ids[idx]} (Person ID: {all_pids[idx]}) | Similarity Score = {scores[idx]:.4f}")

print("\n[+] ✅ SPACETIME-DAMR STANDALONE PIPELINE EXECUTED SUCCESSFULLY!")
